In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from core.training.trajectory import Trajectory

import numpy as np

In [3]:
import numpy as np
import torch
from typing import Tuple

from core.training.trajectory import Trajectory, TrajectoryStep

class TestGame:
    rewards = [[3, -1], [8, 9]]
    epsilon = 0.1
    rng = np.random.default_rng()

    def reward(self, state: int, action: int) -> float:
        return self.rewards[state][action]
    
    def transition(self, state: int, action: int) -> Tuple[int, bool]:
        if self.rng.random() < self.epsilon:
            return (2, True)
        if action == 1:
            return (1 - state, False)
        return (state, False)

    def optimal_policy(self, state: int) -> int:
        if state == 0:
            return 1
        return 0

    def encode(self, state: int) -> torch.Tensor:
        return torch.eye(len(self.rewards))[state]

    def sample_optimal(self, state: int = 0) -> Trajectory:
        trajectory = Trajectory()
        done = False
        while not done:
            action = self.optimal_policy(state)
            reward = self.reward(state, action)
            next_state, done = self.transition(state, action)
            trajectory.append(
                TrajectoryStep(
                    state=self.encode(state),
                    action=action,
                    reward=reward,
                    next_state=None if done else self.encode(next_state),
                    done=done,
                )
            )
            state = next_state
        return trajectory

In [4]:
import torch.nn as nn

value_network = nn.Sequential(
    nn.Linear(2, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
)

def current_values(value_network: nn.Module) -> np.ndarray:
    with torch.no_grad():
        return value_network(torch.eye(2)).squeeze(-1).numpy()

In [5]:
from core.training.trpo.value import ValueTrainer, ValueTrainerConfig
from torch.optim import RMSprop

expected_values = np.array([-1 + 0.81 * 8/0.19, 8/0.19])

game = TestGame()
n_trajectories = 100
trainer = ValueTrainer(gamma=0.9, epsilon=2)
optimizer = RMSprop(value_network.parameters(), lr=1e-3)
trajectories = [game.sample_optimal() for _ in range(n_trajectories)]
config = ValueTrainerConfig(
    optimizer=optimizer,
    trajectories=trajectories,
    value_network=value_network
)
trainer.optimize(config)
current_values(value_network)

array([0.84408045, 0.48235345], dtype=float32)

In [ ]:
for _ in range(500):
    optimizer = RMSprop(value_network.parameters(), lr=1e-4)
    trajectories = [game.sample_optimal() for _ in range(50)]
    config = ValueTrainerConfig(
        optimizer=optimizer,
        trajectories=trajectories,
        value_network=value_network

    )
    trainer.optimize(config)
    print(current_values(value_network))
for _ in range(10):
    optimizer = RMSprop(value_network.parameters(), lr=1e-5)
    trajectories = [game.sample_optimal() for _ in range(10000)]
    config = ValueTrainerConfig(
        optimizer=optimizer,
        trajectories=trajectories,
        value_network=value_network

    )
    trainer.optimize(config)
    print(current_values(value_network))

[0.9169787 0.5390689]
[0.99152076 0.59743494]
[1.0677137 0.6574557]
[1.1455544 0.7191288]
[1.2250413  0.78245306]
[1.3061684  0.84742373]
[1.3889524 0.9140519]
[1.4733746  0.98232377]
[1.5594454 1.0522473]
[1.6471725 1.1238269]
[1.7365429 1.197054 ]
[1.8275561 1.2719268]
[1.9202058 1.3484417]
[2.0145032 1.4266064]
[2.1104462 1.5064197]
[2.2080448 1.5878879]
[2.3072877 1.6710021]
[2.4081712 1.75576  ]
[2.5107028 1.8421677]
[2.6148775 1.9302218]
[2.7206888 2.019916 ]
[2.828153  2.1112628]
[2.937258  2.2042534]
[3.0480058 2.298889 ]
[3.1604042 2.3951764]
[3.274442 2.493104]
[3.3901231 2.5926785]
[3.507461  2.6939063]
[3.6264312 2.7967696]
[3.7470438 2.9012783]
[3.8693    3.0074358]
[3.9932008 3.1152365]
[4.118756  3.2246888]
[4.245943 3.335777]
[4.374779  3.4485142]
[4.505253  3.5628948]
[4.637363 3.678913]
[4.7711267 3.7965808]
[4.9065332 3.9158955]
[5.0435734 4.036843 ]
[5.1822615 4.1594405]
[5.3225827 4.283676 ]
[5.46456   4.4095645]
[5.6081543 4.5370736]
[5.7534122 4.6662474]
[5.90032

In [ ]:
expected_values - current_values(value_network)

array([-0.4076763 ,  0.53871958])